# N102 · 完全、多重与分组背包

**目标：** 根据可重复次数和对象顺序选择转移与循环顺序。

**先修：** N101, N098。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 正序容量；有限次数；二进制分组；分组选择；组合/排列计数。

**配套讲解来源：** [同名逐章意见](../../comment/102_unbounded_bounded_group_knapsack.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

上一章（N101）的 0/1 背包规定每件物品只能拿一次。这一章放宽出三种新玩法，外加一个"计数语义"的重要辨析：

1. **完全背包（零钱兑换）**：每种硬币有无限多枚，问凑出金额 amount 最少要几枚。比如硬币 `[1,2]`、金额 5，答案是 `3`（2+2+1）；硬币 `[2]`、金额 3，答案是 `-1`，因为无论怎么凑 2 的倍数都到不了 3。
2. **组合数与排列数（两种计数）**：同样是"用 1 和 2 凑 3"，如果不在乎顺序，方案只有 2 种（1+1+1、1+2）；如果在乎顺序（把 1+2 和 2+1 视为不同），方案有 3 种（1+1+1、1+2、2+1）。本章用两个函数分别算这两种数：`coin_change_combinations([1,2],3)==2`，`ordered_sum_count([1,2],3)==3`。
3. **多重背包（有限次数）**：每件物品有数量上限，比如"重 2、值 3、最多 5 件"。`bounded_knapsack` 解这种题。
4. **分组背包**：物品分成若干组，每组最多选一件。比如一组里有两件都是"重 1"的物品，值分别是 5 和 6，容量 2，答案只能是 `6`（同组两件不能都拿），这正是 “自动测试”部分 里的那条断言。

这一章的标题级考点是：**循环的内外顺序和正序逆序，直接决定 DP 的语义**。写对循环，比背转移方程更重要。

## 2. 基础知识：先读懂这些词

- **完全背包**：每种物品可取任意多件。一维滚动时容量要**正序**遍历（从小到大），这样更新 `dp[a]` 时读到的 `dp[a-c]` 是本层刚算好的值，允许同一物品继续叠加使用——恰好与 0/1 背包的逆序相反。
- **多重背包**：每种物品最多取 q 件，介于 0/1 和完全之间。
- **二进制分组（打包）**：把"最多 q 件"拆成若干捆：1 件一捆、2 件一捆、4 件一捆……最后剩下的零头单独一捆。这些捆做子集加法可以凑出 0..q 的任何数量，于是每捆当成一件 0/1 物品（重量和价值按比例放大）跑逆序 0/1 背包即可。比如 q=5 拆成 1+2+2，你要 3 件就选"1 捆+2 捆"。这样把 O(q) 次循环压成 O(log q) 捆。
- **分组背包**：每组内最多选一件。转移时必须从**上一组的表**读值，绝不能读本组刚更新的值，否则同组两件会被同时选中。
- **组合数（无序）**：`1+2` 和 `2+1` 算同一种方案。实现要点是**外层枚举币种、内层枚举金额**——币种一个接一个引入，每种硬币的地位被固定下来，方案不会因排列不同被重复计数。
- **排列数 / 有序序列数**：`1+2` 和 `2+1` 算两种方案。实现要点反过来，**外层枚举金额、内层枚举末项**——按"最后一个数是谁"划分方案，自然把不同顺序分开数。
- **正无穷哨兵 `float('inf')`**：在最少硬币数问题里表示"这个金额目前凑不出来"，min 转移会自动避开它；最后还是无穷就返回 −1。
- **`_positive_unique`（正值去重）**：辅助函数，要求面值严格为正并去重排序。正值条件不是走过场——面值为 0 会导致"无限枚硬币金额不变"，计数会无限循环或翻倍；负值会破坏金额递增的依赖方向，DP 直接失效。

## 3. 思路推导：从小例子开始

**问题 1：最少硬币数（`coin_change`）**

- Step 1：定义 `dp[a]` = 凑出金额 a 最少需要的硬币数。`dp[0]=0`（什么都不用），其余初始化为正无穷（还凑不出）。
- Step 2：金额 a 从 1 到 amount 逐个算。最后一枚硬币一定是某种面值 c（要求 c≤a），去掉它之后剩下金额 a−c，所以 `dp[a] = min(dp[a], dp[a-c]+1)`，对所有可用面值取最小。
- Step 3：终态还是无穷说明凑不出，按题意返回 −1。

手算 `coin_change([1,2],5)`：dp[1]=dp[0]+1=1；dp[2]=min(dp[1]+1, dp[0]+1)=1；dp[3]=min(dp[2]+1, dp[1]+1)=2；dp[4]=min(dp[3]+1, dp[2]+1)=2；dp[5]=min(dp[4]+1, dp[3]+1)=3。答案 3，对应 2+2+1。

**问题 2：组合数（`coin_change_combinations`）与排列数（`ordered_sum_count`）**

- Step 1（组合）：外层按面值逐个引入。处理面值 c 之前，`dp[a]` 是"只用更小面值凑 a"的方案数；处理时 `dp[a] += dp[a-c]`（金额正序），含义是"新方案里至少用一枚 c"。因为面值是逐个引入的，每个方案里 c 的使用都发生在它被引入的那一轮，方案之间不会因顺序不同重复。
- Step 2（排列）：外层按金额，内层枚举"最后一个数是谁"。`dp[a] = sum(dp[a-x] for x in nums)`，即以每个可能的末项各分一批，1+2 和 2+1 的末项不同，各自被数一次。

“运行示例”部分 用 `[1,2]` 把两种计数并排放在一起，从金额 0 到 6：

| 目标 | [1,2]组合数 | [1,2]有序序列数 |
|---|---|---|
| 0 | 1 | 1 |
| 1 | 1 | 1 |
| 2 | 2 | 2 |
| 3 | 2 | 3 |
| 4 | 3 | 5 |
| 5 | 3 | 8 |
| 6 | 4 | 13 |

你可以看到金额到 3 之后两列开始分道扬镳：组合数是"几个 2 就几个选择"的 floor(a/2)+1 慢慢涨；有序数却是斐波那契式地 2、3、5、8、13 翻着涨，因为顺序排列让方案数爆炸。理解了这张表，你就理解了两个循环顺序的差别。

**问题 3：多重背包（`bounded_knapsack`）**

- Step 1：对每件 (weight, value, count)，把 count 拆成 1、2、4、… 件一捆的二进制捆（如 5=1+2+2）。
- Step 2：每捆当成一件"重量 weight×k、价值 value×k"的 0/1 物品，跑逆序容量循环。
- Step 3：捆的子集能拼出 0..count 的每种取用量，所以"最多 count 件"的所有用量都被覆盖。

**问题 4：分组背包（`group_knapsack`）**

- Step 1：`dp[c]` = 考虑完前面所有组、容量 c 的最大价值。
- Step 2：进入新组时先把旧表复制一份存为 `prev`，新表 `dp` 从 `prev` 的拷贝开始。转移 `dp[c]=max(dp[c], prev[c-w]+v)` 读取的是**上一组**的 `prev`，这样同组内无论扫过多少件，它们彼此看不到对方的更新，天然保证"每组最多一件"。

## 4. 核心代码：coin_change

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def coin_change(coins, amount):
    coins = _positive_unique(coins)
    dp = [0] + [float('inf')] * amount
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a:
                dp[a] = min(dp[a], dp[a - c] + 1)
    return -1 if dp[amount] == float('inf') else dp[amount]
```

### 逐段读懂代码

**辅助函数：`_positive_unique`**

```python
def _positive_unique(values):
    if any((x <= 0 for x in values)):
        raise ValueError('strictly positive values required')
    return sorted(set(values))
```

- 有任何一个值 ≤0 就报错：0 面值让金额原地打转，负面值让依赖方向成环，DP 都会崩，所以要当场挡住。
- `sorted(set(values))` 去重再排序。去重是为了计数不重复（同面值两枚等价），排序让遍历确定、可复现。

**函数一：`coin_change`**

```python
def coin_change(coins, amount):
    coins = _positive_unique(coins)
    dp = [0] + [float('inf')] * amount
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a:
                dp[a] = min(dp[a], dp[a - c] + 1)
    return -1 if dp[amount] == float('inf') else dp[amount]
```

- `dp=[0]+[float('inf')]*amount`：下标 0..amount；金额 0 零枚可凑，其余先当作凑不出。
- 外层金额、内层面值：对每个金额问"最后一枚是谁最划算"。
- `if c<=a`：面值超过当前金额就跳过，这行是防越界访问（`dp[a-c]` 下标为负）的守卫。
- 末行三元判断：终态还是无穷就返回 −1，否则返回最少枚数。

**函数二：`coin_change_combinations`（组合数）**

```python
def coin_change_combinations(coins, amount):
    dp = [1] + [0] * amount
    for c in _positive_unique(coins):
        for a in range(c, amount + 1):
            dp[a] += dp[a - c]
    return dp[amount]
```

- `dp=[1]+[0]*amount`：凑 0 有且只有"什么都不选"这 1 种。
- **外层面值、内层金额**，且金额从小到大正序：正序让 `dp[a-c]` 可以包含"已经用了面值 c"的方案，即允许 c 用多枚；外层面值固定了每种硬币的"引入时机"，方案不因顺序重复。
- 循环从 c 开始是因为金额小于 c 时根本用不上这种面值。

**函数三：`ordered_sum_count`（有序序列数）**

```python
def ordered_sum_count(nums, target):
    nums = _positive_unique(nums)
    dp = [1] + [0] * target
    for a in range(1, target + 1):
        dp[a] = sum((dp[a - x] for x in nums if x <= a))
    return dp[target]
```

- 与组合数相比**内外层正好对调**：外层金额、内层枚举每个可作末项的数。
- `dp[a-x]` 表示"末项选定为 x，前面的部分凑出 a−x 的有序方案数"，求和后就是把每个末项分支的方案加起来。
- 同样的 `if x<=a` 守卫挡掉越界。

**函数四：`bounded_knapsack`（多重背包，二进制打包）**

```python
def bounded_knapsack(items, capacity):
    dp = [0] * (capacity + 1)
    for weight, value, count in items:
        if weight <= 0 or count < 0:
            raise ValueError('invalid bounded item')
        bundle = 1
        while count:
            take = min(bundle, count)
            w = weight * take
            v = value * take
            for c in range(capacity, w - 1, -1):
                dp[c] = max(dp[c], dp[c - w] + v)
            count -= take
            bundle *= 2
    return dp[-1]
```

- 注释写明了物品格式：(正重量, 价值, 非负数量)，非法值当场报错。
- `bundle=1` 起步、每轮 `bundle*=2`：捆的大小依次是 1、2、4、…；`take=min(bundle,count)` 处理最后一捆的零头（比如 count=5 时三捆是 1、2、2）。
- `w=weight*take; v=value*take`：一捆 take 件的总重与总价值。
- 内层 `range(capacity,w-1,-1)` 是标准 0/1 背包**逆序**——每捆只许拿一次。
- `count-=take` 把已打包的数量扣掉，`while count` 直到全部打包。

**函数五：`group_knapsack`（分组背包）**

```python
def group_knapsack(groups, capacity):
    dp = [0] * (capacity + 1)
    for group in groups:
        prev = dp
        dp = prev[:]
        for w, v in group:
            for c in range(w, capacity + 1):
                dp[c] = max(dp[c], prev[c - w] + v)
    return dp[-1]
```

- `prev=dp; dp=prev[:]`：`prev` 指向上一组的表（只读），`dp` 是它的拷贝（新组在这上面更新）。这一对赋值就是"读旧层、写新层"。
- 转移读的是 `prev[c-w]` 而不是 `dp[c-w]`：同组的其他物品发生在 `prev` 之后的世界里，读不到，所以同组最多选一件。
- 注意这里容量是正序扫，但因为读的是 `prev`，正序逆序都不影响正确性；换成读 `dp` 就错了。

## 5. 分段实现：按顺序运行

**本章接口：** `coin_change(coins, amount)`、`coin_change_combinations(coins, amount)`、`ordered_sum_count(nums, target)`、`bounded_knapsack(items, capacity)`

### _positive_unique

In [1]:
def _positive_unique(values):
    if any((x <= 0 for x in values)):
        raise ValueError('strictly positive values required')
    return sorted(set(values))

### coin_change

In [2]:
def coin_change(coins, amount):
    coins = _positive_unique(coins)
    dp = [0] + [float('inf')] * amount
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a:
                dp[a] = min(dp[a], dp[a - c] + 1)
    return -1 if dp[amount] == float('inf') else dp[amount]

### coin_change_combinations

In [3]:
def coin_change_combinations(coins, amount):
    dp = [1] + [0] * amount
    for c in _positive_unique(coins):
        for a in range(c, amount + 1):
            dp[a] += dp[a - c]
    return dp[amount]

### ordered_sum_count

In [4]:
def ordered_sum_count(nums, target):
    nums = _positive_unique(nums)
    dp = [1] + [0] * target
    for a in range(1, target + 1):
        dp[a] = sum((dp[a - x] for x in nums if x <= a))
    return dp[target]

### bounded_knapsack

In [5]:
def bounded_knapsack(items, capacity):
    dp = [0] * (capacity + 1)
    for weight, value, count in items:
        if weight <= 0 or count < 0:
            raise ValueError('invalid bounded item')
        bundle = 1
        while count:
            take = min(bundle, count)
            w = weight * take
            v = value * take
            for c in range(capacity, w - 1, -1):
                dp[c] = max(dp[c], dp[c - w] + v)
            count -= take
            bundle *= 2
    return dp[-1]

### group_knapsack

In [6]:
def group_knapsack(groups, capacity):
    dp = [0] * (capacity + 1)
    for group in groups:
        prev = dp
        dp = prev[:]
        for w, v in group:
            for c in range(w, capacity + 1):
                dp[c] = max(dp[c], prev[c - w] + v)
    return dp[-1]

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [7]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

show_table(['目标','[1,2]组合数','[1,2]有序序列数'],[(a,coin_change_combinations([1,2],a),ordered_sum_count([1,2],a)) for a in range(7)])

目标,"[1,2]组合数","[1,2]有序序列数"
0,1,1
1,1,1
2,2,2
3,2,3
4,3,5
5,3,8
6,4,13


## 7. 正确性、适用条件与复杂度

无序组合按最后一种可使用的币值分组，不会把同一组硬币的排列重复计数；有序序列按最后一个数划分。二进制数量包的子集能表达0..原数量的每个取用量；有重复表达也不影响取最大值，但不能直接用于方案数计数。

**代价：** 完全背包O(nA)时间O(A)空间；有限次数O(C∑log(q+1))时间O(C)空间；分组O(C∑组内物品数)。amount/target/capacity非负，正值条件防止无限方案与循环依赖。

### 展开理解

**组合数为什么对**：我们按"方案里使用的最大面值是哪种"给所有凑法分组。外层循环一种面值一轮，`dp[a] += dp[a-c]` 发生的时机固定在面值 c 那一轮，一个方案（比如 1+2+2）只会在"轮到 2 时用掉 2 的部分、之前用掉 1 的部分"这一条路径上被计入一次。同一个硬币集合不管你以什么顺序摆，在"面值逐个引入"的框架下只对应唯一的一条计算路径，所以不会重复，也不会漏掉。

**有序数为什么对**：按"序列的最后一个数是谁"划分所有序列，最后一个数不同的序列显然不同，最后一个数相同的序列去掉末项后是独立的子问题，方案数相加即可。这和组合数的分类标准不同，所以数出来的口径也不同（第三节那张表就是证据）。

**二进制打包为什么对**：1、2、4、…、零头，这些捆的任意子集之和能覆盖 0 到 count 之间的每一个数（二进制表示的性质），所以"取用 k 件"总能由若干捆精确拼出。有的取用量可能有不只一种拼法（比如零头捆与前面的捆组合），这对求最大值无所谓——同一种取用量被多条路径表达，max 不受重复影响；但 “正确性与复杂度”部分 特别提醒：**不能把这个技巧直接搬到方案数计数上**，重复表达会让方案被数多次。

**分组背包为什么对**：每组内的合法选择是"一件都不选"或"恰好选其中某一件"。转移从上一组的表出发，把每件物品的"选它"方案叠加到初值（一件都不选，即 `prev` 的拷贝）上，而所有"选它"读的都是 `prev`，互不叠加，恰好实现"组内最多一件"。

**复杂度**：完全背包类（coin_change、两个计数）是 面值数 × 金额，即 O(nA) 时间、O(A) 空间，比如 3 种硬币凑 1000 元就是三千多次操作，眨眼完成。多重背包是 容量 × ∑log(q+1)，即每件物品贡献 log 档而不是 q 档：同样 q=1024，直接拆要 1024 轮，打包只要约 10 捆。分组背包是 容量 × 所有组内物品总数。所有函数都只用一维数组，空间 O(C)。注意这些都和上一章一样是伪多项式——跟数值大小走，不跟输入文本长度走。

## 8. 单元测试：每个用例在检查什么

```python
assert coin_change_combinations([1, 2, 5], 5) == 4
```

- 正常值测试：凑 5 的组合是 5、2+2+1、2+1+1+1、1×1×1×1×1 共 4 种，验证"外层面值"的组合语义没有把排列混进来。

```python
assert ordered_sum_count([1, 2], 3) == 3 and coin_change_combinations([1, 2], 3) == 2
```

- 对拍式双断言：同一个输入 `[1,2]` 凑 3，有序是 3（1+1+1、1+2、2+1），无序是 2（1+1+1、1+2）。两个数并排出现，就是逼你面对两种循环顺序的语义差别。

```python
assert coin_change([2], 3) == -1 and coin_change([], 0) == 0
```

- 第一条是不可达用例：面值全是 2 凑不出奇数 3，检验正无穷哨兵与 −1 的返回契约。
- 第二条是空输入边界：没有硬币但金额也是 0，答案是 0 枚；它同时检验 `_positive_unique([])` 不会报错。

```python
assert group_knapsack([[(1, 5), (1, 6)]], 2) == 6
```

- 分组约束用例：一组两件同重不同值，容量足够装下两件，但同组只能选一件，答案取值大的 6。这条专治"组内读了本组更新"的错误实现——那种实现会返回 11。

```python
for _ in range(150):
    items = [(rng.randrange(1, 5), rng.randrange(1, 8), rng.randrange(5)) for _ in range(3)]
    cap = rng.randrange(12)
    feasible = [sum((t * v for t, (w, v, q) in zip(counts, items))) for counts in product(*(range(q + 1) for w, v, q in items)) if sum((t * w for t, (w, v, q) in zip(counts, items))) <= cap]
    assert bounded_knapsack(items, cap) == max(feasible)
```

- 随机对拍：150 组随机多重背包实例（3 件物品、重量 1..4、价值 1..7、数量 0..4、容量 0..11），暴力枚举每件物品取 0..q 件的所有组合，筛出装得下的，取最大总价值与 DP 对照。数量上限 4 覆盖了"零头捆"的各种形态。

```python
for a in range(12):
    assert coin_change_combinations([1, 2, 5], a) == sum((x + 2 * y + 5 * z == a for x in range(a + 1) for y in range(a + 1) for z in range(a + 1)))
```

- 穷举对拍：金额 0 到 11，暴力三重循环枚举三种硬币的用量，数出凑得恰好等于 a 的组合数与 DP 对照。

In [8]:
assert coin_change_combinations([1, 2, 5], 5) == 4

assert ordered_sum_count([1, 2], 3) == 3 and coin_change_combinations([1, 2], 3) == 2

assert coin_change([2], 3) == -1 and coin_change([], 0) == 0

assert group_knapsack([[(1, 5), (1, 6)]], 2) == 6

from itertools import product

from random import Random

rng = Random(102)

for _ in range(150):
    items = [(rng.randrange(1, 5), rng.randrange(1, 8), rng.randrange(5)) for _ in range(3)]
    cap = rng.randrange(12)
    feasible = [sum((t * v for t, (w, v, q) in zip(counts, items))) for counts in product(*(range(q + 1) for w, v, q in items)) if sum((t * w for t, (w, v, q) in zip(counts, items))) <= cap]
    assert bounded_knapsack(items, cap) == max(feasible)

for a in range(12):
    assert coin_change_combinations([1, 2, 5], a) == sum((x + 2 * y + 5 * z == a for x in range(a + 1) for y in range(a + 1) for z in range(a + 1)))

print('N102: 所有本章断言通过')

N102: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 手写分组背包小例。

**练习 2：** 比较硬币组合数与有序方案数。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（手写分组背包小例）**：提示——自己造两组、每组两三件的小例子（比如两组 `[(2,3),(1,2)]` 和 `[(3,4)]`，容量 5），先在纸上画两张表：一张是"第一组处理完"的 dp，一张是"第二组处理完"的 dp，每格标清它来自 prev 的哪一格。边界别忘了"每组一件都不选"和"容量装不下组里所有件"这两种情形，最后拿 `group_knapsack` 跑一遍对照，也可以手写三层循环暴力枚举每组选哪件来验证。
- **练习 2（组合数 vs 有序方案数）**：提示——选一个两种数开始不同的最小输入（`[1,2]` 凑 3 就是分水岭），把两个函数的每轮 dp 数组手抄下来并排比对：组合数版本一轮面值一轮地更新，有序数版本一轮金额一轮地更新。重点回答两个问题：(1) 组合数实现里如果内层金额改成逆序会发生什么（提示：想想 0/1 背包的逆序在挡什么，这里正序在放行什么）；(2) 有序数实现如果外层换成枚举数、内层换金额，它算出来的会变成哪一种数。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [9]:
# 完整实现：本单元格不依赖其他单元格。
def _positive_unique(values):
    if any((x <= 0 for x in values)):
        raise ValueError('strictly positive values required')
    return sorted(set(values))

def coin_change(coins, amount):
    coins = _positive_unique(coins)
    dp = [0] + [float('inf')] * amount
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a:
                dp[a] = min(dp[a], dp[a - c] + 1)
    return -1 if dp[amount] == float('inf') else dp[amount]

def coin_change_combinations(coins, amount):
    dp = [1] + [0] * amount
    for c in _positive_unique(coins):
        for a in range(c, amount + 1):
            dp[a] += dp[a - c]
    return dp[amount]

def ordered_sum_count(nums, target):
    nums = _positive_unique(nums)
    dp = [1] + [0] * target
    for a in range(1, target + 1):
        dp[a] = sum((dp[a - x] for x in nums if x <= a))
    return dp[target]

def bounded_knapsack(items, capacity):
    dp = [0] * (capacity + 1)
    for weight, value, count in items:
        if weight <= 0 or count < 0:
            raise ValueError('invalid bounded item')
        bundle = 1
        while count:
            take = min(bundle, count)
            w = weight * take
            v = value * take
            for c in range(capacity, w - 1, -1):
                dp[c] = max(dp[c], dp[c - w] + v)
            count -= take
            bundle *= 2
    return dp[-1]

def group_knapsack(groups, capacity):
    dp = [0] * (capacity + 1)
    for group in groups:
        prev = dp
        dp = prev[:]
        for w, v in group:
            for c in range(w, capacity + 1):
                dp[c] = max(dp[c], prev[c - w] + v)
    return dp[-1]

# 示例与回归测试
assert coin_change_combinations([1, 2, 5], 5) == 4

assert ordered_sum_count([1, 2], 3) == 3 and coin_change_combinations([1, 2], 3) == 2

assert coin_change([2], 3) == -1 and coin_change([], 0) == 0

assert group_knapsack([[(1, 5), (1, 6)]], 2) == 6

from itertools import product

from random import Random

rng = Random(102)

for _ in range(150):
    items = [(rng.randrange(1, 5), rng.randrange(1, 8), rng.randrange(5)) for _ in range(3)]
    cap = rng.randrange(12)
    feasible = [sum((t * v for t, (w, v, q) in zip(counts, items))) for counts in product(*(range(q + 1) for w, v, q in items)) if sum((t * w for t, (w, v, q) in zip(counts, items))) <= cap]
    assert bounded_knapsack(items, cap) == max(feasible)

for a in range(12):
    assert coin_change_combinations([1, 2, 5], a) == sum((x + 2 * y + 5 * z == a for x in range(a + 1) for y in range(a + 1) for z in range(a + 1)))

print('N102: 所有本章断言通过')

N102: 所有本章断言通过


## 11. 代表题与迁移

- **322. Coin Change（零钱兑换）**：完全背包求最少枚数，`float('inf')` 哨兵加 −1 契约的原型题，对应 `coin_change`。
- **518. Coin Change II（零钱兑换 II）**：完全背包数组合数，练"外层物品、内层金额正序"这个顺序，对应 `coin_change_combinations`。
- **377. Combination Sum IV（组合总和 IV）**：题目名字叫组合、实际数的是有序序列，练"外层金额、内层枚举末项"的排列计数，对应 `ordered_sum_count`——这题正好是"循环顺序决定语义"的最佳教材。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。